# Week 3 — 03: Model Training

Reads feature-enriched DataFrame from checkpoint 02, builds the clean feature
matrix (drops TXT/pattern metadata), adds clustering features, then trains:

- **Model A** — General AML Detector  (all normal + ALL laundering)
- **Model B** — Patterned-Laundering Specialist (normal + TXT-patterned only)

Saves models to `models/` and checkpoint 03 to `checkpoints/`.


In [1]:
import sys, pathlib

# Resolve project root so notebooks work regardless of CWD or kernel launch dir
_this_nb  = pathlib.Path(__file__) if "__file__" in dir() else pathlib.Path().resolve()
_sentinel = "src"   # directory that marks the project root
_root     = _this_nb.resolve()
for _ in range(10):                          # walk up at most 10 levels
    if (_root / _sentinel).is_dir():
        break
    _root = _root.parent

# Hard-code fallback for direct kernel launch from notebooks/week3/
_fallback = pathlib.Path("/home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project")
if not (_root / _sentinel).is_dir():
    _root = _fallback

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

PROJECT_ROOT = _root
print("Project root:", PROJECT_ROOT)


Project root: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project


In [2]:
from src import state
import numpy as np

df                           = state.load("02_df_features")
train_idx, val_idx, test_idx = state.load("01_indices")
print(f"Loaded: {len(df):,} rows, {len(df.columns)} columns")


Loaded checkpoint: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/02_df_features.pkl
Loaded checkpoint: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/01_indices.pkl
Loaded: 5,078,345 rows, 156 columns


In [ ]:
from src.modeling import build_feature_matrix

X, feature_cols = build_feature_matrix(df, train_idx)
print(f"Feature matrix: {X.shape}")
print(f"Feature count: {len(feature_cols)}")


In [ ]:
from src.modeling import fit_cluster_model, add_cluster_features

scaler, km, cluster_features = fit_cluster_model(X, train_idx)
add_cluster_features(X, train_idx, val_idx, test_idx, scaler, km, cluster_features, feature_cols)
print(f"Total features after clustering: {len(feature_cols)}")


Total features after clustering: 139


In [ ]:
import numpy as np
from src.config import EXTERNAL_PATTERN_COLS

# ── Build A/B population masks ────────────────────────────────────────────────
train_base = np.zeros(len(df), dtype=bool); train_base[train_idx] = True
val_base   = np.zeros(len(df), dtype=bool); val_base[val_idx]     = True
test_base  = np.zeros(len(df), dtype=bool); test_base[test_idx]   = True

patterned   = (df["target"].eq(1) & df["txt_patterned"].eq(1)).to_numpy()
unpatterned = (df["target"].eq(1) & df["txt_patterned"].eq(0)).to_numpy()

train_A_mask = train_base.copy()
val_A_mask   = val_base.copy()
train_B_mask = train_base & ~unpatterned
val_B_mask   = val_base   & ~unpatterned

XA_train = X.loc[train_A_mask, feature_cols].copy()
XA_val   = X.loc[val_A_mask,   feature_cols].copy()
XB_train = X.loc[train_B_mask, feature_cols].copy()
XB_val   = X.loc[val_B_mask,   feature_cols].copy()

yA_train = df.loc[train_A_mask, "target"].to_numpy(dtype=np.int8)
yA_val   = df.loc[val_A_mask,   "target"].to_numpy(dtype=np.int8)
yB_train = df.loc[train_B_mask, "target"].to_numpy(dtype=np.int8)
yB_val   = df.loc[val_B_mask,   "target"].to_numpy(dtype=np.int8)

X_complete_val  = X.loc[val_idx,  feature_cols].copy()
X_complete_test = X.loc[test_idx, feature_cols].copy()
y_complete_val  = df.loc[val_idx,  "target"].to_numpy(dtype=np.int8)
y_complete_test = df.loc[test_idx, "target"].to_numpy(dtype=np.int8)

print(f"Model A TRAIN: {len(yA_train):,} | positives: {int(yA_train.sum()):,}")
print(f"Model B TRAIN: {len(yB_train):,} | positives: {int(yB_train.sum()):,}")


Model A TRAIN: 3,046,810 | positives: 2,849
Model B TRAIN: 3,045,690 | positives: 1,729


In [ ]:
from src.modeling import build_xgb_model
from sklearn.metrics import average_precision_score, roc_auc_score

# ── Train Model A ─────────────────────────────────────────────────────────────
model_A = build_xgb_model(yA_train, feature_cols)
model_A.fit(XA_train, yA_train, eval_set=[(XA_val, yA_val)], verbose=100)

pA_val = model_A.predict_proba(X_complete_val)[:, 1]
print("Model A AP (complete VAL):", average_precision_score(y_complete_val, pA_val))
print("Model A ROC-AUC:", roc_auc_score(y_complete_val, pA_val))


[0]	validation_0-aucpr:0.19949
[100]	validation_0-aucpr:0.48169
[200]	validation_0-aucpr:0.49669
[300]	validation_0-aucpr:0.50128


KeyboardInterrupt: 

In [ ]:
# ── Train Model B ─────────────────────────────────────────────────────────────
model_B = build_xgb_model(yB_train, feature_cols)
model_B.fit(XB_train, yB_train, eval_set=[(XB_val, yB_val)], verbose=100)

pB_val_spec = model_B.predict_proba(XB_val)[:, 1]
pB_val      = model_B.predict_proba(X_complete_val)[:, 1]
print("Model B AP (specialist VAL):", average_precision_score(yB_val, pB_val_spec))
print("Model B AP (complete VAL):",   average_precision_score(y_complete_val, pB_val))


In [ ]:
from src.config import MODELS_DIR

model_A.save_model(MODELS_DIR / "model_A_general_aml.json")
model_B.save_model(MODELS_DIR / "model_B_pattern_specialist.json")

state.save("03_models",       (model_A, model_B))
state.save("03_val_scores",   (pA_val, pB_val, pB_val_spec, y_complete_val, yA_val, yB_val))
state.save("03_matrices",     (X, feature_cols, train_A_mask, val_A_mask, train_B_mask, val_B_mask,
                                X_complete_val, X_complete_test, y_complete_test,
                                train_base, val_base, test_base, patterned, unpatterned,
                                train_idx, val_idx, test_idx))
print("Checkpoint 03 saved.")
